In [2]:
!pip install pyvista numpy pillow trame


   ---------------------------------------- 0/5 [trame-common]
   ---------------- ----------------------- 2/5 [trame-client]
   ---------------- ----------------------- 2/5 [trame-client]
   ------------------------ --------------- 3/5 [trame-server]
   -------------------------------- ------- 4/5 [trame]
   -------------------------------- ------- 4/5 [trame]
   ---------------------------------------- 5/5 [trame]



In [1]:
from pathlib import Path
import pyvista as pv
import numpy as np
from PIL import Image

# ============================================================
# SETTINGS
# ============================================================

# Folder containing STL files
STL_FOLDER = Path(r"C:\Users\vazquez\Documents\TAMP-OS\comparison_output")

# Output folders
OUTPUT_FOLDER = Path("individual_renders")
ZOOM_FOLDER = Path("zoom_renders")

OUTPUT_FOLDER.mkdir(exist_ok=True)
ZOOM_FOLDER.mkdir(exist_ok=True)

# ============================================================
# THEME
# ============================================================

# Choose:
# "dark"  -> dark background
# "light" -> white background

THEME = "dark"

# ============================================================
# AUTOMATIC COLORS
# ============================================================

if THEME == "dark":

    BACKGROUND = "#1e1e1e"

    # darker matte grey
    MESH_COLOR = "#707070"

    TEXT_COLOR = "white"

    LIGHT_INTENSITY = 0.85

elif THEME == "light":

    BACKGROUND = "white"

    # darker mesh for visibility
    MESH_COLOR = "#5a5a5a"

    TEXT_COLOR = "black"

    LIGHT_INTENSITY = 0.65

else:

    raise ValueError(
        "THEME must be 'dark' or 'light'"
    )

# ============================================================
# WINDOW
# ============================================================

WINDOW_SIZE = (1800, 1800)

# ============================================================
# CAMERA
# ============================================================

AZIMUTH = 25
ELEVATION = 20

# IMPORTANT:
# smaller value = zoom OUT more
# this fixes clipped corners

FULL_ZOOM = 0.92

# zoom render
ZOOM_ZOOM = 2.8

# ============================================================
# LIGHTING
# ============================================================

LIGHT_POSITION = (1.5, 0.6, 0.5)

# ============================================================
# LOAD STL FILES
# ============================================================

stl_files = sorted(STL_FOLDER.glob("*.stl"))

if len(stl_files) == 0:
    raise FileNotFoundError(
        f"No STL files found in: {STL_FOLDER}"
    )

print(f"Found {len(stl_files)} STL files")

# ============================================================
# FUNCTION: CREATE PLOTTER
# ============================================================

def make_plotter():

    plotter = pv.Plotter(
        off_screen=True,
        window_size=WINDOW_SIZE,
    )

    plotter.set_background(BACKGROUND)

    return plotter

# ============================================================
# FUNCTION: ADD MESH STYLE
# ============================================================

def add_mesh_style(plotter, mesh):

    plotter.add_mesh(
        mesh,

        color=MESH_COLOR,

        smooth_shading=True,

        # matte appearance
        ambient=0.06,

        diffuse=0.95,

        specular=0.015,

        specular_power=4,

        show_edges=False,
    )

    # --------------------------------------------------------
    # MAIN LIGHT
    # --------------------------------------------------------

    light = pv.Light(
        position=LIGHT_POSITION,
        focal_point=(0, 0, 0),
        intensity=LIGHT_INTENSITY,
        color="white",
    )

    plotter.add_light(light)

    # --------------------------------------------------------
    # SOFT FILL LIGHT
    # --------------------------------------------------------

    fill = pv.Light(
        position=(-1.0, -0.3, 0.2),
        focal_point=(0, 0, 0),
        intensity=0.22,
        color="white",
    )

    plotter.add_light(fill)

# ============================================================
# FUNCTION: APPLY CAMERA
# ============================================================

def set_camera(plotter, zoom):

    plotter.camera_position = "iso"

    plotter.camera.azimuth = AZIMUTH
    plotter.camera.elevation = ELEVATION

    # Orthographic projection
    plotter.enable_parallel_projection()

    plotter.camera.zoom(zoom)

# ============================================================
# FUNCTION: CLEAN LABEL
# ============================================================

def clean_label(path):

    label = (
        path.stem
        .replace("_", " ")
    )

    return label

# ============================================================
# RENDER LOOP
# ============================================================

for stl_path in stl_files:

    print(f"Rendering: {stl_path.name}")

    mesh = pv.read(stl_path)

    label = clean_label(stl_path)

    # ========================================================
    # FULL VIEW
    # ========================================================

    plotter = make_plotter()

    add_mesh_style(plotter, mesh)

    set_camera(plotter, FULL_ZOOM)

    plotter.add_text(
        label,
        position="upper_edge",
        font_size=20,
        color=TEXT_COLOR,
    )

    output_png = OUTPUT_FOLDER / f"{stl_path.stem}.png"

    plotter.screenshot(output_png)

    plotter.close()

    # ========================================================
    # ZOOM VIEW
    # ========================================================

    zoom_plotter = make_plotter()

    add_mesh_style(zoom_plotter, mesh)

    set_camera(zoom_plotter, ZOOM_ZOOM)

    # --------------------------------------------------------
    # CENTER CAMERA
    # --------------------------------------------------------

    bounds = mesh.bounds

    center_x = (bounds[0] + bounds[1]) / 2
    center_y = (bounds[2] + bounds[3]) / 2
    center_z = (bounds[4] + bounds[5]) / 2

    zoom_plotter.camera.focal_point = (
        center_x,
        center_y,
        center_z,
    )

    zoom_plotter.add_text(
        f"{label}\nZOOM",
        position="upper_edge",
        font_size=20,
        color=TEXT_COLOR,
    )

    zoom_png = ZOOM_FOLDER / f"{stl_path.stem}_ZOOM.png"

    zoom_plotter.screenshot(zoom_png)

    zoom_plotter.close()

# ============================================================
# DONE
# ============================================================

print("\nDone!")

print(f"Saved full renders to: {OUTPUT_FOLDER}")

print(f"Saved zoom renders to: {ZOOM_FOLDER}")

Found 12 STL files
Rendering: _DSC9378_blur_0.5.stl
Rendering: _DSC9378_blur_1.0.stl
Rendering: _DSC9378_blur_1.5.stl
Rendering: _DSC9378_blur_2.0.stl
Rendering: _DSC9378_relief_height_1.0.stl
Rendering: _DSC9378_relief_height_2.0.stl
Rendering: _DSC9378_relief_height_3.0.stl
Rendering: _DSC9378_relief_height_5.0.stl
Rendering: _DSC9378_resolution_128.stl
Rendering: _DSC9378_resolution_256.stl
Rendering: _DSC9378_resolution_512.stl
Rendering: _DSC9378_resolution_64.stl

Done!
Saved full renders to: individual_renders
Saved zoom renders to: zoom_renders
